<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Unusual readings in public open data

**[Unusual readings in public open data](https://learning.nextia-ai.com/courses/anomaly-detection/m10/unusual-readings-in-public-open-data/)** · Anomaly Detection: Find What Is Unusual · Case study 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** look for faults in four years of real hourly air readings from 12 stations, with no answer key. You compare four detectors under a review budget of 5 station-days a week, and you evaluate them three ways: faults that you plant, a hand review of a random sample, and the agreement between detectors. Then you write an alert policy and run it on 8 new weeks.

| | |
|---|---|
| **Time** | About 2.5 hours with the lesson page; the notebook runs in under a minute |
| **Needs** | An internet connection for the setup cells (about 8 MB). No account and no key. |
| **You should know** | The MAD and the robust z-score, from [Median, MAD and IQR](https://learning.nextia-ai.com/courses/anomaly-detection/m02/median-mad-and-iqr/). Isolation Forest, from [Isolation Forest](https://learning.nextia-ai.com/courses/anomaly-detection/m04/isolation-forest/). The three ways to evaluate, from [Evaluate without a full answer key](https://learning.nextia-ai.com/courses/anomaly-detection/m07/evaluate-without-a-full-answer-key/). The budget, from [The alert budget](https://learning.nextia-ai.com/courses/anomaly-detection/m07/the-alert-budget/). |
| **Data** | UCI Beijing Multi-Site Air Quality, CC BY 4.0: real hourly PM2.5 and PM10 readings of 12 stations, March 2013 to February 2017 |
| **Tested** | Python 3.12.13 with pandas 3.1.1, NumPy 2.5.4 and scikit-learn 1.9.1, in Jupyter on macOS (Apple silicon, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m10/unusual-readings-in-public-open-data/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M10-L03-unusual-readings-in-public-open-data/unusual-readings-in-public-open-data-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the data

Run the next cell. It makes a new folder `open-air-readings` and downloads four files into it. It checks each file's checksum:

- `beijing-air-quality.zip` (8.2 MB): the hourly readings of 12 air-quality stations, from the UCI Machine Learning Repository;
- `reviews.csv`, `floor-reviews.csv` and `new-reviews.csv`: our hand reviews of the flags, one line per station and day, with the reason.

It needs an internet connection and no account, and takes about 10 seconds. You should see `Ready:` and the four files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `open-air-readings` (in Colab: the **Files** panel on the left), then run the cell again. If UCI changed its file, the cell takes the copy from the course's labs repository instead.

**Source and licence.** "Beijing Multi-Site Air Quality" by Song Chen, UCI Machine Learning Repository, <https://doi.org/10.24432/C5RK5G>, licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The readings are from the Beijing Municipal Environmental Monitoring Center. We use the file as it is ([dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M10-L03-unusual-readings-in-public-open-data/dataset.md)).

In [ ]:
# Setup: download the readings and the reviews into open-air-readings/ and check them. Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

FILES = {
    "beijing-air-quality.zip": [
        [
            "https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip",
            "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/notebooks/M10-L03-unusual-readings-in-public-open-data/beijing-air-quality.zip"
        ],
        "b04da438b2f331ac0ffd45aebdfec0d20d2367feb5f6948c4b1f7ce1191e33c4"
    ],
    "reviews.csv": [
        [
            "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/notebooks/M10-L03-unusual-readings-in-public-open-data/reviews.csv"
        ],
        "0e5026c58ea53ec9ed4ef9b9c3251872f970bfb84a9a3f0e3016d909457d979c"
    ],
    "floor-reviews.csv": [
        [
            "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/notebooks/M10-L03-unusual-readings-in-public-open-data/floor-reviews.csv"
        ],
        "9b9d55c81b072f16256392de6b65ff1a90ca063fa726d157215357fa7ed1c280"
    ],
    "new-reviews.csv": [
        [
            "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/notebooks/M10-L03-unusual-readings-in-public-open-data/new-reviews.csv"
        ],
        "5ec7620d27ebf9e17e373bda8e5adf30aa151020546e410a668ce061e8909f1d"
    ]
}   # name: [[source, second source], SHA-256]
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("open-air-readings").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            return False
    return True

for name, (urls, expected) in FILES.items():
    path = PROJECT / name
    if path.exists() and sha256(path) == expected:
        continue
    if (HERE / name).exists():  # a copy next to the notebook
        shutil.copy(HERE / name, path)
    for url in urls:
        if path.exists() and sha256(path) == expected:
            break
        if not download(url, path):
            print(f"Could not download {url}.")
    if not path.exists():
        raise SystemExit(f"Could not download {name}. Check your internet connection.")
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder open-air-readings and run again.")

os.chdir(PROJECT)
print("Ready:", ", ".join(f"{n} ({Path(n).stat().st_size / 1e6:.2f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The readings

One row of the source is one station in one hour. **PM2.5** is the mass of fine particles in the air, in micrograms per cubic metre (µg/m³). **PM10** includes slightly larger particles, so in physics PM2.5 is a part of PM10. The next cell reads the 12 station files from the zip and makes two tables: one column per station, one row per hour.

> **Check.** You should see `420,768 readings: 12 stations x 35,064 hours, 2013-03-01 to 2017-02-28`.

In [ ]:
import io, zipfile
outer = zipfile.ZipFile("beijing-air-quality.zip")
inner = zipfile.ZipFile(io.BytesIO(outer.read("PRSA2017_Data_20130301-20170228.zip")))
readings = pd.concat([pd.read_csv(inner.open(n), usecols=["year", "month", "day", "hour", "PM2.5", "PM10", "station"])
                      for n in sorted(inner.namelist()) if n.endswith(".csv")], ignore_index=True)
readings["time"] = pd.to_datetime(readings[["year", "month", "day", "hour"]])
pm25 = readings.pivot(index="time", columns="station", values="PM2.5")   # one column per station
pm10 = readings.pivot(index="time", columns="station", values="PM10")
print(f"{len(readings):,} readings: {pm25.shape[1]} stations x {pm25.shape[0]:,} hours, "
      f"{pm25.index.min():%Y-%m-%d} to {pm25.index.max():%Y-%m-%d}")

### First checks

Before any detector, count the problems that you can see without one.

> **Check.** You should see:
>
> ```text
> missing PM2.5 readings: 2.1%; hours with no station at all: 81
> smallest value: 2, read 17 times; the value 3, read 8,810 times; largest: 999
> PM2.5 equal to PM10: 23.5%; PM2.5 above PM10: 5.1%
> readings that are not whole numbers: {'PM2.5': 181, 'PM10': 229}
> stations whose value did not change from 05:00 to 09:00 on 18 September 2013: 12 of 12
> ```

In [ ]:
values = pm25.stack()                       # every PM2.5 reading that is not missing
print(f"missing PM2.5 readings: {pm25.isna().mean().mean():.1%}; hours with no station at all: {pm25.isna().all(axis=1).sum()}")
print(f"smallest value: {values.min():.0f}, read {(values == values.min()).sum():,} times; "
      f"the value 3, read {(values == 3).sum():,} times; largest: {values.max():.0f}")
print(f"PM2.5 equal to PM10: {(pm25 == pm10).stack().mean():.1%}; PM2.5 above PM10: {(pm25 > pm10).stack().mean():.1%}")
same_as_before = pm25.eq(pm25.shift()) & pm25.notna()
decimals = {c: int((readings[c].dropna() % 1 != 0).sum()) for c in ["PM2.5", "PM10"]}
print("readings that are not whole numbers:", decimals)
print("stations whose value did not change from 05:00 to 09:00 on 18 September 2013:",
      int(same_as_before.loc["2013-09-18 05:00":"2013-09-18 09:00"].all().sum()), "of 12")

**What this tells us.** About 2% of the readings are missing. The smallest value that appears often is 3: it looks like the **floor**, the lowest value that the instruments report. PM2.5 equals PM10 in almost a quarter of the hours, and is above it in 5%, so the two analysers at a station do not always agree. A few hundred readings are not whole numbers, unlike all the others. And on 18 September 2013, all 12 stations kept the same value from 04:00 to 09:00: the feed itself froze.

## 2. Frame the problem

Mina publishes these readings on a public air map. She can review **5 station-days a week** (one station on one day is one flag), at about 6 minutes each. Both numbers are our assumptions. We compare the detectors on 52 weeks, March 2015 to February 2016, and keep the last 8 weeks of the data for the end-to-end run.

> **Check.** You should see `evaluation weeks: 52`.

In [ ]:
HISTORY = slice("2013-03-01", "2015-03-01")      # learn what normal looks like
EVALUATION = slice("2015-03-02", "2016-02-28")   # 52 weeks, Monday to Sunday: compare the detectors here
NEW_INPUT = slice("2017-01-02", "2017-02-26")    # 8 weeks the detectors never saw: the end-to-end run
BUDGET = 5                                       # station-days that Mina can review each week (assumption)
MINUTES_PER_REVIEW = 6                           # assumption
print("evaluation weeks:", (pd.Timestamp("2016-02-28") - pd.Timestamp("2015-03-02")).days // 7 + 1)

## 3. The idea: compare a station with its neighbours

When the air gets dirty, it gets dirty for the whole city: every station rises. When one instrument fails, only that station moves. So compare each reading with the **median of the other stations** in the same hour. We use a log scale, log(1 + reading), so that 80 against 90 and 400 against 450 count as the same small gap.

A tiny example: five stations in one hour. Station E reads 240, and the median of the other four is 82.5.

> **Check.** You should see a gap of about 0 for A to D, and 1.06 for E.

In [ ]:
# One hour at five stations, in micrograms per cubic metre.
hour = pd.Series({"A": 80, "B": 85, "C": 78, "D": 90, "E": 240})
others = {s: hour.drop(s).median() for s in hour.index}      # the median of the other four
gap = np.log1p(hour) - np.log1p(pd.Series(others))           # how far above its neighbours, on a log scale
pd.DataFrame({"reading": hour, "median of the others": others, "gap (log)": gap.round(2)})

Now two real nights. On the left, the New Year fireworks of 8 February 2016: every station rises, a real event. On the right, 16 January 2017: one station drops to 3 for two hours while the other 11 stay near 90, a fault.

> **Check.** You should see two charts. On the left, three lines rise together after midnight. On the right, the Wanshouxigong line drops to 10 and 3 at 11:00 and 12:00.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.4))
night = pm25.loc["2016-02-07 18:00":"2016-02-08 12:00"]             # Chinese New Year: the fireworks
left.plot(night.index, night.max(axis=1), label="highest station")
left.plot(night.index, night.median(axis=1), label="median of the 12")
left.plot(night.index, night.min(axis=1), label="lowest station")
left.set_title("A real event: every station rises"); left.set_ylabel("PM2.5"); left.legend()
day = pm25.loc["2017-01-16 06:00":"2017-01-16 18:00"]
right.plot(day.index, day["Wanshouxigong"], label="Wanshouxigong")
right.plot(day.index, day.drop(columns="Wanshouxigong").median(axis=1), label="median of the other 11")
right.set_title("A fault: one station drops alone"); right.legend()
for ax in (left, right):
    ax.tick_params(axis="x", labelrotation=30)
plt.tight_layout(); plt.show()

The next cell computes the **neighbour gap** for every station and hour, and two helpers: the robust z-score from the MAD lesson, and a count of hours without a change.

> **Check.** You should see a median gap of 0.0, and 95% of the gaps between -1.04 and 0.7 in the history.

In [ ]:
def neighbour_gap(pm25, min_others=6):
    """log(1 + reading) minus the median of the other stations in the same hour."""
    x = np.log1p(pm25)
    gap = {}
    for s in x.columns:
        others = x.drop(columns=s)
        gap[s] = (x[s] - others.median(axis=1)).where(others.notna().sum(axis=1) >= min_others)
    return pd.DataFrame(gap)

def robust_z(values, reference):
    """(value - median) / (1.4826 x MAD), with the median and MAD of the reference period: the MAD lesson's score."""
    median = reference.median()
    mad = (reference - median).abs().median() * 1.4826
    return (values - median) / mad

def repeats(pm25):
    """How many hours in a row the value has been the same as the hour before (0 = it changed)."""
    same = pm25.eq(pm25.shift()) & pm25.notna()
    return same.apply(lambda c: c.groupby((~c).cumsum()).cumsum())

gap = neighbour_gap(pm25)
print("median gap in the history:", gap.loc[HISTORY].stack().median().round(3),
      "| 95% of gaps between", gap.loc[HISTORY].stack().quantile([0.025, 0.975]).round(2).tolist())

## 4. Six detectors

- **Own MAD (baseline):** the robust z-score of a reading against its own station's last 7 days. It knows nothing about the other stations.
- **Rules:** written before we looked: the same value for 6 hours or more (above the floor), PM2.5 more than twice PM10 plus 50, or the maximum, 999. A rule says 1 or 0.
- **Neighbour z:** the robust z-score of the neighbour gap, with each station's own median and MAD from the history.
- **Isolation Forest** on five numbers per hour: the level, the neighbour gap, the change since the last hour, PM2.5 against PM10, and the hours without a change. It learns from 60,000 hours of the history.
- **Rules, then neighbour z** and **rules, then Isolation Forest:** a rule flag first, then the score. The rules help both in the same way, so the comparison is fair.

> **Check.** You should see `trained Isolation Forest on 60,000 hours of the history`.

In [ ]:
from sklearn.ensemble import IsolationForest

def own_mad(pm25):
    """Baseline: the robust z-score of each reading against its own station's last 7 days (past hours only)."""
    x = np.log1p(pm25)
    median = x.rolling(168, min_periods=100).median().shift(1)
    mad = (x - median).abs().rolling(168, min_periods=100).median().shift(1) * 1.4826
    return ((x - median) / mad).abs()

def rules(pm25, pm10):
    """1 when a written rule fires: stuck for 6 hours or more above the floor, far above PM10, or at the maximum."""
    stuck = (repeats(pm25) >= 5) & (pm25 > 3)
    above_pm10 = pm25 > 2 * pm10 + 50
    at_maximum = pm25 >= 999
    return (stuck | above_pm10 | at_maximum).astype(float).where(pm25.notna())

def neighbour_z(pm25, history=HISTORY):
    """The robust z-score of the neighbour gap, with each station's median and MAD from the history."""
    g = neighbour_gap(pm25)
    return pd.DataFrame({s: robust_z(g[s], g.loc[history, s]) for s in g.columns}).abs()

def hourly_features(pm25, pm10):
    """One row per station and hour: the level, the neighbour gap, the change since the last hour, PM2.5 against
    PM10, and the hours without a change."""
    x = np.log1p(pm25)
    parts = {"level": x, "gap": neighbour_gap(pm25), "change": x.diff(), "vs_pm10": x - np.log1p(pm10),
             "repeats": repeats(pm25).clip(upper=24).where(pm25.notna())}
    return pd.concat({k: v.stack(future_stack=True) for k, v in parts.items()}, axis=1).dropna()

train = hourly_features(pm25, pm10).loc[HISTORY].sample(60_000, random_state=0)
forest = IsolationForest(n_estimators=200, random_state=0).fit(train)

def isolation_forest(pm25, pm10):
    f = hourly_features(pm25, pm10)
    return pd.Series(-forest.score_samples(f), index=f.index).unstack()

def rules_then_neighbour(pm25, pm10):
    """A rule flag first (score 1,000 or more), then the neighbour z-score for the rest."""
    fired, z = rules(pm25, pm10).fillna(0), neighbour_z(pm25)
    return z.where(fired == 0, 1000 + z.fillna(0))

def rules_then_forest(pm25, pm10):
    """A rule flag first, then the Isolation Forest score: the same help from the rules, for a fair comparison."""
    fired, s = rules(pm25, pm10).fillna(0), isolation_forest(pm25, pm10)
    return s.where(fired == 0, 1000 + s.fillna(0))

DETECTORS = {"own MAD (baseline)": lambda a, b: own_mad(a), "rules": rules, "neighbour z": lambda a, b: neighbour_z(a),
             "Isolation Forest": isolation_forest, "rules, then neighbour z": rules_then_neighbour,
             "rules, then Isolation Forest": rules_then_forest}
print("trained Isolation Forest on", f"{len(train):,}", "hours of the history")

Each detector scores every hour. A station-day takes its highest hourly score, and each week Mina gets the 5 highest station-days.

> **Check.** You should see 260 flags in 52 weeks for each detector, except the rules: they fired on only 4 station-days.

In [ ]:
def station_days(hourly, period):
    """The score of a station-day is its highest hourly score."""
    h = hourly.loc[period]
    day = h.groupby(h.index.normalize()).max().stack().rename("score").reset_index()
    day.columns = ["day", "station", "score"]
    return day

def weekly_flags(hourly, period, budget=BUDGET, start=None):
    """The `budget` highest station-days of each week (ties in time order). A 0/1 rule sends only its 1s."""
    day = station_days(hourly, period)
    day["week"] = (day["day"] - pd.Timestamp(start or period.start)).dt.days // 7
    day = day.sort_values(["week", "score", "day", "station"], ascending=[True, False, True, True])
    flags = day.groupby("week").head(budget)
    return flags[flags["score"] > 0].reset_index(drop=True)

clean_scores = {name: d(pm25, pm10) for name, d in DETECTORS.items()}
clean_flags = {name: weekly_flags(s, EVALUATION) for name, s in clean_scores.items()}
pd.Series({name: len(f) for name, f in clean_flags.items()}, name="flags in 52 weeks")

## 5. Way 1: plant faults that you know

The data has no answer key. So we make a small one: we copy the readings and plant 104 faults of four kinds in the evaluation weeks, 2 a week. A **spike** (one hour far too high), a **stuck** sensor (the same value for 8 hours), a **scale** error (every reading of the day 60% too high) and a **clock** error (the readings of 6 hours earlier).

> **Check.** You should see 26 faults of each kind.

In [ ]:
def plant_faults(pm25, seed=34):
    """Plant 104 faults in the evaluation weeks, 2 a week, at random stations and days: 26 of each kind."""
    rng = np.random.default_rng(seed)
    planted, rows = pm25.copy(), []
    kinds = rng.permutation(["spike", "stuck", "scale", "clock"] * 26)
    weeks = pd.date_range(EVALUATION.start, EVALUATION.stop, freq="7D")
    for w, week in enumerate(weeks):
        used = set()
        for kind in kinds[2 * w: 2 * w + 2]:
            while True:                                   # a station-day with at least 20 readings, one per station
                day = week + pd.Timedelta(days=int(rng.integers(7)))
                station = pm25.columns[int(rng.integers(12))]
                hours = pd.date_range(day, periods=24, freq="h")
                if station not in used and pm25.loc[hours, station].notna().sum() >= 20:
                    break
            used.add(station)
            v = pm25.loc[hours, station]
            if kind == "spike":                           # one hour far too high
                h = hours[int(rng.integers(24))]
                planted.loc[h, station] = 3 * v[h] + 150 if pd.notna(v[h]) else 300
            elif kind == "stuck":                         # the same value for 8 hours
                s = int(rng.integers(16))
                planted.loc[hours[s:s + 8], station] = v.iloc[s:].dropna().iloc[0]
            elif kind == "scale":                         # every reading of the day 60% too high
                planted.loc[hours, station] = (v * 1.6).round()
            else:                                         # "clock": the readings of 6 hours earlier
                planted.loc[hours, station] = pm25.loc[hours - pd.Timedelta(hours=6), station].to_numpy()
            rows.append({"station": station, "day": day, "kind": kind})
    return planted, pd.DataFrame(rows)

pm25_planted, planted = plant_faults(pm25)
planted_scores = {name: d(pm25_planted, pm10) for name, d in DETECTORS.items()}
planted_flags = {name: weekly_flags(s, EVALUATION) for name, s in planted_scores.items()}
planted.kind.value_counts()

> **Predict.** Which detector finds the most stuck sensors? Which finds the most scale errors? Write your guesses down, then run the next cell.

The cell counts, for each detector and each kind, the share of planted faults that are among the 5 weekly flags. This is **recall at the budget**.

> **Check.** You should see `rules, then neighbour z` with 0.62 in the column `all`, Isolation Forest with 0.45 and the baseline with 0.10.

In [ ]:
def found(flags, planted=planted):
    """For each planted fault: is its station-day among the weekly flags?"""
    keys = set(zip(flags["station"], flags["day"]))
    return pd.Series([(s, d) in keys for s, d in zip(planted["station"], planted["day"])], index=planted.index)

recall = pd.DataFrame({name: found(f).groupby(planted["kind"]).mean() for name, f in planted_flags.items()}).T
recall["all"] = [found(f).mean() for f in planted_flags.values()]
recall.round(2)

The rules find every stuck sensor, and Isolation Forest finds every spike. No detector finds more than 15% of the scale errors: a reading 60% too high still looks like air. Planted faults only test the kinds that we planted.

## 6. Way 2: review a random sample by hand

We take 20 random flags of three detectors from the evaluation weeks (with no planted faults), and all 4 flags of the rules. For each one, we look at the data around its worst hour and write a verdict:

- **fault:** the PM2.5 reading is wrong by about 10 µg/m³ or more: it is at the floor or stuck while its own PM10 or the other stations are clearly higher, or it jumps for an hour and nothing else does;
- **real:** the air changed: PM10 moves with PM2.5, or other stations show the same change a little earlier or later;
- **normal:** nothing that a reader of the map would see, for example 3 against 9 in clean air;
- **unclear:** the evidence points both ways.

> **Check.** You should see `64 picked flags, 61 different station-days to review (366 minutes at 6 minutes each)`.

In [ ]:
# The hand review: 20 random flags of each detector, from the evaluation weeks without planted faults.
rng = np.random.default_rng(2015)
picked = []
for name in ["own MAD (baseline)", "neighbour z", "Isolation Forest"]:
    f = clean_flags[name]
    for i in rng.choice(len(f), 20, replace=False):
        picked.append({"detector": name, "station": f.at[i, "station"], "day": f.at[i, "day"]})
picked += [{"detector": "rules", "station": s, "day": d}                 # the rules fired only 4 times: review all 4
           for s, d in zip(clean_flags["rules"]["station"], clean_flags["rules"]["day"])]
picked = pd.DataFrame(picked)
to_review = picked.drop_duplicates(["station", "day"])
print(f"{len(picked)} picked flags, {len(to_review)} different station-days to review "
      f"({len(to_review) * MINUTES_PER_REVIEW} minutes at {MINUTES_PER_REVIEW} minutes each)")

This is the evidence card for one flag. Read it: the station reads 3 for two hours, and so does its PM10, while the other stations read 82 to 92. The hours before are missing. Our verdict: a fault.

In [ ]:
def evidence(station, day, hourly=None):
    """The data a reviewer looks at for one flag: the worst hour, with 3 hours on each side."""
    day = pd.Timestamp(day)
    hourly = clean_scores["neighbour z"] if hourly is None else hourly
    worst = hourly.loc[str(day.date()), station].idxmax()
    hours = pd.date_range(worst - pd.Timedelta(hours=3), worst + pd.Timedelta(hours=3), freq="h")
    others = pm25.loc[hours].drop(columns=station)
    card = pd.DataFrame({"PM2.5": pm25.loc[hours, station], "PM10": pm10.loc[hours, station],
                         "others, median": others.median(axis=1), "others, highest": others.max(axis=1),
                         "same as before": repeats(pm25).loc[hours, station]})
    card.index = hours.strftime("%m-%d %H:00")
    return card

evidence("Wanliu", "2015-10-19")

The file `reviews.csv` has our verdict and reason for each of the 61 station-days. The next cell counts the faults per detector, with the 95% interval from [Evaluate without a full answer key](https://learning.nextia-ai.com/courses/anomaly-detection/m07/evaluate-without-a-full-answer-key/).

> **Check.** You should see:
>
> ```text
> {'fault': 23, 'real': 18, 'normal': 16, 'unclear': 4}
> own MAD (baseline)   faults  5 of 20: 0.25 (95% interval 0.06 to 0.44) | real 7, normal 7, unclear 1
> neighbour z          faults 10 of 20: 0.50 (95% interval 0.28 to 0.72) | real 7, normal 0, unclear 3
> Isolation Forest     faults  9 of 20: 0.45 (95% interval 0.23 to 0.67) | real 3, normal 7, unclear 1
> rules                faults  1 of 4: 0.25 (95% interval 0.00 to 0.67) | real 1, normal 2, unclear 0
> ```

In [ ]:
reviews = pd.read_csv("reviews.csv", parse_dates=["day"])     # our verdicts, one per station-day, with the reason
print(reviews.verdict.value_counts().to_dict())
judged = picked.merge(reviews, on=["station", "day"], how="left")

def share_with_interval(hits, n):
    p = hits / n
    margin = 1.96 * np.sqrt(p * (1 - p) / n)
    return f"{hits:2d} of {n}: {p:.2f} (95% interval {max(0, p - margin):.2f} to {min(1, p + margin):.2f})"

for name, g in judged.groupby("detector", sort=False):
    counts = g.verdict.value_counts()
    print(f"{name:20} faults {share_with_interval(counts.get('fault', 0), len(g))} | "
          f"real {counts.get('real', 0)}, normal {counts.get('normal', 0)}, unclear {counts.get('unclear', 0)}")

Twenty reviews give an interval about 0.4 wide. Neighbour z (10 faults of 20) and Isolation Forest (9 of 20) cannot be told apart. The baseline is clearly worse: most of its flags are real changes or clean air.

## 7. Way 3: agreement between detectors

Agreement needs no answer key. The **Jaccard index** is the flags in both lists, divided by the flags in either list.

> **Check.** You should see 32 station-days flagged by all three, and an agreement of 0.23 between neighbour z and Isolation Forest.

In [ ]:
def keys(flags):
    return set(zip(flags["station"], flags["day"]))
names = ["own MAD (baseline)", "neighbour z", "Isolation Forest"]
agreement = pd.DataFrame({a: {b: len(keys(clean_flags[a]) & keys(clean_flags[b])) / len(keys(clean_flags[a]) | keys(clean_flags[b]))
                              for b in names} for a in names}).round(2)
all_three = keys(clean_flags[names[0]]) & keys(clean_flags[names[1]]) & keys(clean_flags[names[2]])
print("flagged by all three:", len(all_three))
agreement

The detectors agree little. They look at different things, and each one sees faults that the others miss. Low agreement is not bad news here: it says that one detector alone would miss a lot.

### Failure case: the baseline flags clean air

The baseline compares each reading with its own last 7 days on a log scale. In clean air, 3 against 8 is a big step on that scale. The next cell shows one of its flags.

> **Check.** You should see readings of 3 to 8 at Wanliu, with the other stations at 5 to 9.

In [ ]:
# The baseline's flag on 22 January 2016 at Wanliu: what did it see?
evidence("Wanliu", "2016-01-22", clean_scores["own MAD (baseline)"])

Nothing is wrong here: the whole city has clean air. **Fix:** compare with the neighbours, not with the station's own past. **Consequence:** in the review, 14 of the baseline's 20 flags were clean air or real changes of the air. Its budget goes to flags with no fault in them, and real faults wait.

## 8. What the review taught us: drops to the floor

Most faults in the review look the same: PM2.5 falls to the floor for an hour or two, while its PM10 and the other stations stay higher. A new rule can find them: **a reading of 4 or less while the median of the other stations is 15 or more**.

> **Check.** You should see 224 floor-drop station-days (4.3 a week), and a table where 18 of the 23 reviewed faults are floor drops.

In [ ]:
def floor_drops(pm25, low=4, others_at_least=15):
    """Readings at the floor (4 or less) while the median of the other stations is 15 or more."""
    others = pd.DataFrame({s: pm25.drop(columns=s).median(axis=1) for s in pm25.columns})
    return (pm25 <= low) & (others >= others_at_least)

drops = floor_drops(pm25)
drop_days = drops.groupby(drops.index.normalize()).any().stack()
reviews["floor drop"] = [bool(drop_days.get((d, s), False)) for d, s in zip(reviews["day"], reviews["station"])]
print("floor-drop station-days in the evaluation weeks:", int(drop_days.loc[EVALUATION].sum()),
      f"({drop_days.loc[EVALUATION].sum() / 52:.1f} a week)")
pd.crosstab(reviews["verdict"], reviews["floor drop"])

The 18 faults came from the sample of flags, not from the rule. To know how often the rule itself is right, we review a new random sample of 20 of its 224 station-days.

In [ ]:
# A second review: 20 random floor-drop station-days of the evaluation weeks, to estimate the rule's precision.
days = drop_days.loc[EVALUATION]
days = days[days].index.to_frame(index=False, name=["day", "station"])
floor_picked = days.iloc[np.random.default_rng(2016).choice(len(days), 20, replace=False)].reset_index(drop=True)
print(f"{len(floor_picked)} of {len(days)} floor-drop station-days picked for review")

> **Check.** You should see 13 faults of 20: 0.65, with an interval from 0.44 to 0.86.

In [ ]:
floor_reviews = pd.read_csv("floor-reviews.csv", parse_dates=["day"])
floor_judged = floor_picked.merge(floor_reviews, on=["station", "day"], how="left")
print("floor drops reviewed:", floor_judged.verdict.value_counts().to_dict())
print("faults", share_with_interval((floor_judged.verdict == "fault").sum(), len(floor_judged)))

About two in three floor drops are faults. The others are clean air at one station. Hiding a low reading costs little: the map shows "no reading" instead of "good air", and it never hides pollution. So this rule can act **with no review**, and Mina's budget goes to the rest.

## 9. The policy, and the comparison table

The policy: (1) floor drops become missing readings, with no review; (2) the rules, then the neighbour z-score, for the 5 weekly flags.

> **Check.** You should see `planted faults found by the policy: 0.62`, and a table with one row per detector.

In [ ]:
def policy(pm25, pm10):
    """Step 1: floor drops become missing readings, with no review. Step 2: rules, then the neighbour z-score."""
    cleaned = pm25.mask(floor_drops(pm25))
    return rules_then_neighbour(cleaned, pm10)

DETECTORS["policy"] = policy
planted_scores["policy"] = policy(pm25_planted, pm10)
planted_flags["policy"] = weekly_flags(planted_scores["policy"], EVALUATION)
print("planted faults found by the policy:", round(found(planted_flags["policy"]).mean(), 2))

In [ ]:
def precision_reviewed(name):
    g = judged[judged.detector == name]
    return round(float((g.verdict == "fault").mean()), 2) if len(g) else None

rows = []
for name in DETECTORS:
    n = len(planted_flags[name])
    rows.append({"detector": name, "flags a week": round(n / 52, 1),
                 "planted found": round(found(planted_flags[name]).mean(), 2),
                 "planted missed a week": round((~found(planted_flags[name])).sum() / 52, 1),
                 "reviewed: faults": precision_reviewed(name),
                 "review minutes a week": round(n / 52 * MINUTES_PER_REVIEW)})
comparison = pd.DataFrame(rows).set_index("detector")
shared = keys(clean_flags["rules, then neighbour z"]) & keys(clean_flags["neighbour z"])
print(f"rules, then neighbour z shares {len(shared)} of its {len(clean_flags['rules, then neighbour z'])} flags with neighbour z")
comparison

**What this tells us.** The simple methods win. Rules, then neighbour z find 64 of the 104 planted faults; rules, then Isolation Forest find 60. The two are close, but the neighbour z-score is easier to explain: "this station reads 3, its neighbours read 90". The table's review column comes from 20 reviews each, so it has wide intervals.

## 10. End to end: 8 new weeks

Now run the policy on January and February 2017, which no step has seen. No faults are planted. Each flag gets a reason from its data.

> **Check.** You should see `8 weeks: 54 floor-drop station-days removed with no review; 40 flags to review, 0 of them from a rule`.

In [ ]:
# The end-to-end run: the policy on 8 new weeks, with no planted faults.
new_scores = policy(pm25, pm10)
new_flags = weekly_flags(new_scores, NEW_INPUT)
new_drops = drop_days.loc[NEW_INPUT]

def reason(station, day, scores=new_scores):
    """One line that explains a flag with the data of its worst hour."""
    t = scores.loc[str(pd.Timestamp(day).date()), station].idxmax()
    row = evidence(station, day, scores).loc[t.strftime("%m-%d %H:00")]
    why = []
    if row["same as before"] >= 5 and row["PM2.5"] > 3:
        why.append(f"the same value for {int(row['same as before']) + 1} hours")
    if row["PM2.5"] > 2 * row["PM10"] + 50:
        why.append(f"PM2.5 {row['PM2.5']:.0f} is far above its PM10, {row['PM10']:.0f}")
    if row["PM2.5"] >= 999:
        why.append("the instrument's maximum")
    if not why:
        side = "above" if row["PM2.5"] > row["others, median"] else "below"
        why.append(f"{row['PM2.5']:.0f}, {side} the median of the others, {row['others, median']:.0f}; its PM10 {row['PM10']:.0f}")
    return f"{t:%H:00} " + "; ".join(why)

new_flags["reason"] = [reason(s, d) for s, d in zip(new_flags["station"], new_flags["day"])]
print(f"8 weeks: {int(new_drops.sum())} floor-drop station-days removed with no review; "
      f"{len(new_flags)} flags to review, {(new_flags.score >= 1000).sum()} of them from a rule")
pd.set_option("display.max_colwidth", 100)
new_flags[["day", "station", "reason"]].head(10)

Mina reviews all 40 flags (`new-reviews.csv`).

> **Check.** You should see 7 faults of 40: 0.17.

In [ ]:
new_reviews = pd.read_csv("new-reviews.csv", parse_dates=["day"])   # Mina reviewed every flag of the 8 weeks
new_judged = new_flags.merge(new_reviews, on=["station", "day"], how="left")
print(new_judged.verdict.value_counts().to_dict())
print("faults", share_with_interval((new_judged.verdict == "fault").sum(), len(new_judged)))

Only 7 of the 40 are faults. The floor rule took the easy faults first, and in winter the queue fills with **real** local differences: clean air reaches the northern stations first, or one district stays polluted longer. Those flags are not wasted: Mina keeps the readings and learns where the city differs. But the review budget buys fewer faults than the evaluation weeks promised.

The New Year night of 28 January 2017 is the city's dirtiest night in these weeks. Does the policy call it a fault?

> **Check.** You should see the 12 stations' median above 500, the baseline with 1 flag on that day and the policy with 0.

In [ ]:
# Chinese New Year 2017 (28 January): the night of fireworks. Which detector flags it?
night = slice("2017-01-28 00:00", "2017-01-28 06:00")
print("median of the 12 stations, 00:00 to 06:00:", pm25.loc[night].median(axis=1).round().astype(int).tolist())
for name in ["own MAD (baseline)", "policy"]:
    f = weekly_flags(DETECTORS[name](pm25, pm10), NEW_INPUT)
    n = int((f.day == "2017-01-28").sum())
    print(f"{name}: {n} flag{'' if n == 1 else 's'} on 28 January")

## 11. Check yourself

**Your turn 1 (change one thing).** Neighbour z found 12% of the planted scale errors with 5 flags a week. With a budget of **10** flags a week, what share of the scale errors does it find? Use `weekly_flags(planted_scores["neighbour z"], EVALUATION, budget=10)` and `found()`, round to 2 decimals, and write it as `share_scale_neighbour_10`.

In [ ]:
share_scale_neighbour_10 = ...   # your code

In [ ]:
expect('share_scale_neighbour_10', share_scale_neighbour_10, 0.23, tolerance=0.001)

**Your turn 2.** The floor rule asks for a median of 15 or more at the other stations. Make it stricter: 30 or more. How many station-days of the evaluation weeks have a floor drop then? Use `floor_drops(pm25, others_at_least=30)` and write the count as `floor_days_30`.

In [ ]:
floor_days_30 = ...   # your code

In [ ]:
expect('floor_days_30', floor_days_30, 84)

## Recap

- With no labels, you evaluated detectors three ways: planted faults (exact, but only for the kinds you planted), a hand review of random flags (honest, but wide intervals), and agreement (free, but it cannot see what all miss).
- Comparing a station with its neighbours separates real events from faults better than comparing it with its own past.
- Rules, then the neighbour z-score beat Isolation Forest on the planted faults, and the review could not tell them apart.
- The review found the most common fault, drops to the floor. A rule now handles them with no review, because its mistakes are cheap.
- On new weeks, the review queue held more real local differences than faults: check a policy on new data before you promise a number.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Control charts on a real production line](https://learning.nextia-ai.com/courses/anomaly-detection/m10/control-charts-on-a-real-production-line/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m10/unusual-readings-in-public-open-data/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The readings are real public data; the people in the story are invented.